In [1]:
import os
os.chdir("/root/Projects/SecureRAG")
print(os.getcwd())

/root/Projects/SecureRAG


In [2]:
from transformers.retrieval_rag import RagRetriever


# retriever = RagRetriever.from_pretrained("models/rag-sequence-nq")

retriever = RagRetriever.from_pretrained(
    "models/rag-sequence-nq",
    index_name="legacy",
    index_path="wiki_dpr"
)

In [3]:
retriever.init_retrieval()

In [4]:
from transformers.tokenization_rag import RagTokenizer

from securerag.eval import RagSequenceForGeneration

rag_seq = RagSequenceForGeneration.from_pretrained(
    "models/rag-sequence-nq", retriever=retriever
)

/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/modeling_utils.py:927: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(resolve

In [22]:
tokenizer = RagTokenizer.from_pretrained("models/rag-sequence-nq")
question_encoder = rag_seq.question_encoder

In [23]:
from securerag import data
from securerag.config import Config

cfg = Config()
debug_data = data.load("data/open_domain_data/TQA/test.json", 1000)

In [24]:
question = debug_data[0]["question"]
question

'Who was the man behind The Chipmunks?'

In [25]:
input_dict = tokenizer.prepare_seq2seq_batch(question, return_tensors="pt")
input_ids = input_dict["input_ids"]
input_mask = input_dict["attention_mask"]

In [26]:
question_enc_outputs = question_encoder(
    input_ids, attention_mask=input_mask, return_dict=True
)
question_encoder_last_hidden_state = question_enc_outputs[
    0
]  # hidden states of question encoder

In [27]:
import torch


retriever_outputs = retriever(
    input_ids,
    question_encoder_last_hidden_state.cpu().detach().to(torch.float32).numpy(),
    prefix="",
    n_docs=100,
    return_tensors="pt",
)

In [28]:
doc_ids = retriever_outputs["doc_ids"]

In [29]:
doc_ids

tensor([[ 2258499,  4666356,  1365774,  2258565,  2258534,  2258551,  2258501,
          2462248,  2258528,  2258541,  2258505, 11005431,  2462252,  2258553,
          4666362,  6516852,  2258525, 10758995,  2258522,  5915579,  4666358,
          2462249, 15200540,  5179633,  1365782,  2258500,  2258511,  1687076,
          3914686,  1365775, 13395348,  2258510,  3242431, 10244823, 19377437,
           852370,  4666361,  2269395,  7748371, 12368145,  2258517, 10244817,
          3650649, 10244822,  2258514, 19377436, 20603442,  2258546,  5110832,
          2258508,  2258559,  2258555, 10758999,  2258535,  2258503,  4435598,
         15780698, 11687531,  8767495,  2258548,  3673338, 12478980,  2258521,
          2258557, 12368157,   962302,  7087863,  4435602,  1472051, 10958648,
         17192877,  9558627,  1687077,   827208,  3676978, 17991862,  3959749,
          3724300, 10958644,  4236587,  3010770,  2258554,  2258515, 17823190,
          2258544,  2258526,  4635606,   827192,  22

In [31]:
embs = retriever_outputs["retrieved_doc_embeds"].squeeze(0)
torch.mm(question_encoder_last_hidden_state, embs.transpose(1, 0))

RuntimeError: mat2 must be a matrix

In [19]:
import torch
def get_scores(question):
    input_dict = tokenizer.prepare_seq2seq_batch(question, return_tensors="pt")
    input_ids = input_dict["input_ids"]
    input_mask = input_dict["attention_mask"]
    question_enc_outputs = question_encoder(
        input_ids, attention_mask=input_mask, return_dict=True
    )
    question_encoder_last_hidden_state = question_enc_outputs[
        0
    ]  # hidden states of question encoder
    (embs, doc_ids, doc_dicts) = retriever_outputs = retriever.retrieve(
        question_encoder_last_hidden_state.cpu().detach().to(torch.float32).numpy(),
        n_docs=100,
    )
    doc_ids = doc_ids[0]
    doc_dict = doc_dicts[0]
    embs = torch.Tensor(embs)
    embs = embs.squeeze(0)
    scores = torch.mm(question_encoder_last_hidden_state, embs.transpose(1, 0))[0].tolist()
    titles = doc_dict["title"]
    texts = doc_dict["text"]
    ret = [
        {
            "id": int(doc_id),
            "title": titles[i],
            "text": texts[i],
            "score": scores[i]
        }
        for i, doc_id in enumerate(doc_ids)
    ]
    return ret

In [20]:
tmp = get_scores(question)
print(tmp[0])

{'id': 7624371, 'title': 'Linda Davis', 'text': 'Linda Davis Linda Kaye Davis (born November 26, 1962) is an American country music singer. Before beginning a career as a solo artist, she had three minor country singles in the charts as one half of the duo Skip & Linda. In her solo career, Davis has recorded five studio albums for major record labels and more than 15 singles. Her highest chart entry is "Does He Love You", her 1993 duet with Reba McEntire, which reached number one on the "Billboard" country charts and won both singers the Grammy for Best Country Vocal Collaboration. Her highest solo chart position', 'score': 89.24541473388672}


In [21]:
import json

from tqdm import tqdm

output_path = "data/open_domain_data/TQA/test_with_scores.json"
input_path = "data/open_domain_data/TQA/test.json"

examples = []
with open(input_path, "r") as fin, open(output_path, "w") as fout:
    json_data = json.load(fin)
    for k, example in tqdm(enumerate(json_data)):
        question = example["question"]
        example["ctxs"] = get_scores(question)
        examples.append(example)
    json.dump(examples, fout)

11313it [47:53,  3.94it/s]
